In [1]:
import sys
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')
# Add parent directory
sys.path.append(os.path.dirname(os.getcwd()))

# Import utilities
from utils.data_loader import UniversalDataLoader

# Set display options
pd.set_option('display.max_columns', 100)
pd.set_option('display.max_rows', 50)


print("✓ All imports successful!")

✓ All imports successful!


In [2]:
# Initialize with your data path
loader = UniversalDataLoader(
    data_path='../data/raw/',
    verbose=True,
    sample_rows=10
)

In [3]:
# Load everything with auto-detection
all_data = loader.load_all(parse_dates=True)


📂 Loading 11 CSV files from: ..\data\raw

✓ Loaded account_statuses: 3 rows, 2 columns (parsed: )
  Columns: AccountStatusID, StatusName
✓ Loaded account_types: 5 rows, 2 columns (parsed: )
  Columns: AccountTypeID, TypeName
   ✓ Confident date: OpeningDate (success rate: 100.0%)
   ✓ Parsed OpeningDate with format: auto
✓ Loaded accounts: 1,667 rows, 6 columns (parsed: OpeningDate)
  Columns: AccountID, CustomerID, AccountTypeID, AccountStatusID, Balance, OpeningDate
✓ Loaded addresses: 1,222 rows, 4 columns (parsed: )
  Columns: AddressID, Street, City, Country
✓ Loaded branches: 50 rows, 3 columns (parsed: )
  Columns: BranchID, BranchName, AddressID
✓ Loaded customer_types: 3 rows, 2 columns (parsed: )
  Columns: CustomerTypeID, TypeName
   ✓ Confident date: DateOfBirth (success rate: 100.0%)
   ✓ Parsed DateOfBirth with format: auto
✓ Loaded customers: 1,111 rows, 6 columns (parsed: DateOfBirth)
  Columns: CustomerID, FirstName, LastName, DateOfBirth, AddressID, CustomerTypeID
✓ 

In [12]:
# Check what was detected
loader.print_summary()


📊 LOADING SUMMARY
✅ Tables loaded: 11
❌ Errors: 0

📁 Table Details:
  • account_statuses: 3 rows × 2 cols
  • account_types: 5 rows × 2 cols
  • accounts: 1,667 rows × 6 cols (parsed: OpeningDate)
  • addresses: 1,222 rows × 4 cols
  • branches: 50 rows × 3 cols
  • customer_types: 3 rows × 2 cols
  • customers: 1,111 rows × 6 cols (parsed: DateOfBirth)
  • loan_statuses: 3 rows × 2 cols
  • loans: 333 rows × 7 cols (parsed: StartDate, EstimatedEndDate)
  • transaction_types: 4 rows × 2 cols (parsed: TransactionTypeID)
  • transactions: 50,000 rows × 8 cols (parsed: TransactionID, TransactionTypeID, TransactionDate)


# CLEAN ALL TABLES

In [13]:
# Create a copy of the original data
cleaned_data = {}
cleaning_log = {}

print("\n📋 Starting with raw data...")

for table_name, df in all_data.items():
    if df is not None:
        cleaned_data[table_name] = df.copy()
        cleaning_log[table_name] = {
            'original_rows': len(df),
            'original_cols': len(df.columns),
            'actions': []
        }
        print(f"   ✓ Copied {table_name}: {len(df):,} rows, {len(df.columns)} cols")


📋 Starting with raw data...
   ✓ Copied account_statuses: 3 rows, 2 cols
   ✓ Copied account_types: 5 rows, 2 cols
   ✓ Copied accounts: 1,667 rows, 6 cols
   ✓ Copied addresses: 1,222 rows, 4 cols
   ✓ Copied branches: 50 rows, 3 cols
   ✓ Copied customer_types: 3 rows, 2 cols
   ✓ Copied customers: 1,111 rows, 6 cols
   ✓ Copied loan_statuses: 3 rows, 2 cols
   ✓ Copied loans: 333 rows, 7 cols
   ✓ Copied transaction_types: 4 rows, 2 cols
   ✓ Copied transactions: 50,000 rows, 8 cols


## HANDLE MISSING VALUES

In [5]:
for table_name, df in cleaned_data.items():
    if df is None:
        continue
    
    missing_counts = df.isnull().sum()
    missing_cols = missing_counts[missing_counts > 0]
    
    if len(missing_cols) == 0:
        print(f"\n✅ {table_name}: No missing values")
        continue
    
    print(f"\n📋 {table_name.upper()}:")
    print(f"   Missing values found: {missing_cols.to_dict()}")
    
    # Handle each column based on type
    for col in missing_cols.index:
        missing_count = missing_cols[col]
        missing_pct = (missing_count / len(df)) * 100
        
        # Determine column type
        is_numeric = pd.api.types.is_numeric_dtype(df[col])
        is_date = 'datetime' in str(df[col].dtype)
        is_categorical = df[col].dtype == 'object' or df[col].dtype.name == 'category'
        
        # Strategy based on column type and missing percentage
        if missing_pct > 50:
            # Drop column if more than 50% missing
            df.drop(columns=[col], inplace=True)
            cleaning_log[table_name]['actions'].append(f"Dropped column '{col}' ({missing_pct:.1f}% missing)")
            print(f"   ✅ Dropped column '{col}' ({missing_pct:.1f}% missing)")
            
        elif is_date:
            # For dates, drop rows with missing dates
            before = len(df)
            df.dropna(subset=[col], inplace=True)
            dropped = before - len(df)
            cleaning_log[table_name]['actions'].append(f"Dropped {dropped} rows with missing '{col}'")
            print(f"   ✅ Dropped {dropped} rows with missing '{col}'")
            
        elif is_numeric:
            # Fill numeric with median
            median_val = df[col].median()
            df[col].fillna(median_val, inplace=True)
            cleaning_log[table_name]['actions'].append(f"Filled '{col}' with median ({median_val:.2f})")
            print(f"   ✅ Filled '{col}' with median ({median_val:.2f})")
            
        elif is_categorical:
            # Fill categorical with mode or 'Unknown'
            mode_val = df[col].mode()
            if not mode_val.empty:
                fill_val = mode_val[0]
            else:
                fill_val = 'Unknown'
            df[col].fillna(fill_val, inplace=True)
            cleaning_log[table_name]['actions'].append(f"Filled '{col}' with '{fill_val}'")
            print(f"   ✅ Filled '{col}' with '{fill_val}'")
        
        else:
            # Fallback: fill with 'Unknown'
            df[col].fillna('Unknown', inplace=True)
            cleaning_log[table_name]['actions'].append(f"Filled '{col}' with 'Unknown'")
            print(f"   ✅ Filled '{col}' with 'Unknown'")

print("\n✅ Missing values handled!")


✅ account_statuses: No missing values

✅ account_types: No missing values

📋 ACCOUNTS:
   Missing values found: {'OpeningDate': 33}
   ✅ Dropped 33 rows with missing 'OpeningDate'

📋 ADDRESSES:
   Missing values found: {'Street': 24, 'City': 26, 'Country': 24}
   ✅ Filled 'Street' with 'Unknown'
   ✅ Filled 'City' with 'Unknown'
   ✅ Filled 'Country' with 'Unknown'

✅ branches: No missing values

✅ customer_types: No missing values

📋 CUSTOMERS:
   Missing values found: {'FirstName': 22, 'LastName': 23, 'DateOfBirth': 32}
   ✅ Filled 'FirstName' with 'Unknown'
   ✅ Filled 'LastName' with 'Unknown'
   ✅ Dropped 32 rows with missing 'DateOfBirth'

✅ loan_statuses: No missing values

📋 LOANS:
   Missing values found: {'StartDate': 6, 'EstimatedEndDate': 6}
   ✅ Dropped 6 rows with missing 'StartDate'
   ✅ Dropped 6 rows with missing 'EstimatedEndDate'

✅ transaction_types: No missing values

📋 TRANSACTIONS:
   Missing values found: {'TransactionDate': 1000}
   ✅ Dropped 1000 rows with mi

## REMOVE DUPLICATES

In [6]:
for table_name, df in cleaned_data.items():
    if df is None:
        continue
    
    dup_count = df.duplicated().sum()
    
    if dup_count == 0:
        print(f"\n✅ {table_name}: No duplicates found")
        continue
    
    print(f"\n📋 {table_name.upper()}:")
    print(f"   Found {dup_count:,} duplicate rows")
    
    # Remove duplicates
    before = len(df)
    df.drop_duplicates(inplace=True)
    removed = before - len(df)
    
    cleaning_log[table_name]['actions'].append(f"Removed {removed} duplicate rows")
    print(f"   ✅ Removed {removed:,} duplicate rows")

print("\n✅ Duplicates removed!")


✅ account_statuses: No duplicates found

✅ account_types: No duplicates found

📋 ACCOUNTS:
   Found 16 duplicate rows
   ✅ Removed 16 duplicate rows

📋 ADDRESSES:
   Found 12 duplicate rows
   ✅ Removed 12 duplicate rows

✅ branches: No duplicates found

✅ customer_types: No duplicates found

📋 CUSTOMERS:
   Found 11 duplicate rows
   ✅ Removed 11 duplicate rows

✅ loan_statuses: No duplicates found

📋 LOANS:
   Found 3 duplicate rows
   ✅ Removed 3 duplicate rows

✅ transaction_types: No duplicates found

📋 TRANSACTIONS:
   Found 490 duplicate rows
   ✅ Removed 490 duplicate rows

✅ Duplicates removed!


# HANDLE OUTLIERS

In [7]:
# Tables that should skip outlier detection (small/reference tables)
skip_outliers = ['branches', 'addresses', 'customer_types', 'account_types', 
                 'account_statuses', 'loan_statuses', 'transaction_types']

for table_name, df in cleaned_data.items():
    if df is None:
        continue
    
    if table_name in skip_outliers:
        print(f"\n✅ {table_name}: Skipping outlier detection (small/reference table)")
        continue
    
    # Get numeric columns
    numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    
    if len(numeric_cols) == 0:
        print(f"\n✅ {table_name}: No numeric columns to check")
        continue
    
    print(f"\n📋 {table_name.upper()}:")
    
    outliers_found = False
    
    for col in numeric_cols:
        # Calculate IQR
        Q1 = df[col].quantile(0.25)
        Q3 = df[col].quantile(0.75)
        IQR = Q3 - Q1
        
        if IQR == 0:
            continue
        
        lower_bound = Q1 - 1.5 * IQR
        upper_bound = Q3 + 1.5 * IQR
        
        # Find outliers
        outlier_mask = (df[col] < lower_bound) | (df[col] > upper_bound)
        outlier_count = outlier_mask.sum()
        
        if outlier_count > 0:
            outliers_found = True
            print(f"   {col}: Found {outlier_count:,} outliers")
            
            # Cap outliers
            before_min = df[col].min()
            before_max = df[col].max()
            
            df.loc[df[col] < lower_bound, col] = lower_bound
            df.loc[df[col] > upper_bound, col] = upper_bound
            
            after_min = df[col].min()
            after_max = df[col].max()
            
            print(f"      Capped from [{before_min:.2f}, {before_max:.2f}] to [{after_min:.2f}, {after_max:.2f}]")
            cleaning_log[table_name]['actions'].append(f"Capped {outlier_count} outliers in '{col}'")
    
    if not outliers_found:
        print(f"   ✅ No outliers detected")

print("\n✅ Outliers handled!")


✅ account_statuses: Skipping outlier detection (small/reference table)

✅ account_types: Skipping outlier detection (small/reference table)

📋 ACCOUNTS:
   ✅ No outliers detected

✅ addresses: Skipping outlier detection (small/reference table)

✅ branches: Skipping outlier detection (small/reference table)

✅ customer_types: Skipping outlier detection (small/reference table)

📋 CUSTOMERS:
   ✅ No outliers detected

✅ loan_statuses: Skipping outlier detection (small/reference table)

📋 LOANS:
   ✅ No outliers detected

✅ transaction_types: Skipping outlier detection (small/reference table)

📋 TRANSACTIONS:
   ✅ No outliers detected

✅ Outliers handled!


## DISPLAY CLEANING SUMMARY

In [8]:
summary_data = []

for table_name, df in cleaned_data.items():
    if df is None:
        continue
    
    original_rows = all_data[table_name].shape[0] if all_data.get(table_name) is not None else 0
    original_cols = all_data[table_name].shape[1] if all_data.get(table_name) is not None else 0
    
    remaining_missing = df.isnull().sum().sum()
    remaining_duplicates = df.duplicated().sum()
    
    # Check for remaining outliers
    numeric_cols = df.select_dtypes(include=[np.number]).columns
    outliers_remaining = 0
    for col in numeric_cols:
        Q1 = df[col].quantile(0.25)
        Q3 = df[col].quantile(0.75)
        IQR = Q3 - Q1
        if IQR != 0:
            lower = Q1 - 1.5 * IQR
            upper = Q3 + 1.5 * IQR
            outliers_remaining += len(df[(df[col] < lower) | (df[col] > upper)])
    
    summary_data.append({
        'Table': table_name,
        'Original_Rows': original_rows,
        'Original_Cols': original_cols,
        'Final_Rows': len(df),
        'Final_Cols': len(df.columns),
        'Rows_Removed': original_rows - len(df),
        'Cols_Removed': original_cols - len(df.columns),
        'Missing_Remaining': remaining_missing,
        'Duplicates_Remaining': remaining_duplicates,
        'Outliers_Remaining': outliers_remaining,
        'Actions': len(cleaning_log[table_name]['actions'])
    })

summary_df = pd.DataFrame(summary_data)
print(summary_df.to_string(index=False))

            Table  Original_Rows  Original_Cols  Final_Rows  Final_Cols  Rows_Removed  Cols_Removed  Missing_Remaining  Duplicates_Remaining  Outliers_Remaining  Actions
 account_statuses              3              2           3           2             0             0                  0                     0                   0        0
    account_types              5              2           5           2             0             0                  0                     0                   0        0
         accounts           1667              6        1618           6            49             0                  0                     0                   0        2
        addresses           1222              4        1210           4            12             0                 72                     0                   0        4
         branches             50              3          50           3             0             0                  0                     0          

## DETAILED CLEANING LOG

In [9]:
for table_name, log in cleaning_log.items():
    if log['actions']:
        print(f"\n📋 {table_name.upper()}:")
        for action in log['actions']:
            print(f"   • {action}")
    else:
        print(f"\n📋 {table_name.upper()}: No cleaning needed")


📋 ACCOUNT_STATUSES: No cleaning needed

📋 ACCOUNT_TYPES: No cleaning needed

📋 ACCOUNTS:
   • Dropped 33 rows with missing 'OpeningDate'
   • Removed 16 duplicate rows

📋 ADDRESSES:
   • Filled 'Street' with 'Unknown'
   • Filled 'City' with 'Unknown'
   • Filled 'Country' with 'Unknown'
   • Removed 12 duplicate rows

📋 BRANCHES: No cleaning needed

📋 CUSTOMER_TYPES: No cleaning needed

📋 CUSTOMERS:
   • Filled 'FirstName' with 'Unknown'
   • Filled 'LastName' with 'Unknown'
   • Dropped 32 rows with missing 'DateOfBirth'
   • Removed 11 duplicate rows

📋 LOAN_STATUSES: No cleaning needed

📋 LOANS:
   • Dropped 6 rows with missing 'StartDate'
   • Dropped 6 rows with missing 'EstimatedEndDate'
   • Removed 3 duplicate rows

📋 TRANSACTION_TYPES: No cleaning needed

📋 TRANSACTIONS:
   • Dropped 1000 rows with missing 'TransactionDate'
   • Removed 490 duplicate rows


# SAVE CLEANED DATA

In [11]:
output_path = '../data/processed/'


# Save as CSV
for name, df in cleaned_data.items():
    if df is not None:
        filepath = os.path.join(output_path, f"{name}.csv")
        df.to_csv(filepath, index=False)
        print(f"   ✓ Saved {name} to {filepath}")

# Save summary report
summary_df.to_csv(os.path.join(output_path, 'cleaning_summary.csv'), index=False)
print(f"\n   ✓ Saved cleaning summary to {output_path}cleaning_summary.csv")

   ✓ Saved account_statuses to ../data/processed/account_statuses.csv
   ✓ Saved account_types to ../data/processed/account_types.csv
   ✓ Saved accounts to ../data/processed/accounts.csv
   ✓ Saved addresses to ../data/processed/addresses.csv
   ✓ Saved branches to ../data/processed/branches.csv
   ✓ Saved customer_types to ../data/processed/customer_types.csv
   ✓ Saved customers to ../data/processed/customers.csv
   ✓ Saved loan_statuses to ../data/processed/loan_statuses.csv
   ✓ Saved loans to ../data/processed/loans.csv
   ✓ Saved transaction_types to ../data/processed/transaction_types.csv
   ✓ Saved transactions to ../data/processed/transactions.csv

   ✓ Saved cleaning summary to ../data/processed/cleaning_summary.csv


In [14]:
for table_name, df in cleaned_data.items():
    if df is None:
        print(f"❌ {table_name:15s} → Failed to load")
        continue
    
    print(f"✅ {table_name:15s} → {df.shape[0]:>6,} rows × {df.shape[1]:>3} cols")
    print(f"   Columns: {', '.join(df.columns.tolist())}")
    print("-" * 70)

print("="*70)

✅ account_statuses →      3 rows ×   2 cols
   Columns: AccountStatusID, StatusName
----------------------------------------------------------------------
✅ account_types   →      5 rows ×   2 cols
   Columns: AccountTypeID, TypeName
----------------------------------------------------------------------
✅ accounts        →  1,667 rows ×   6 cols
   Columns: AccountID, CustomerID, AccountTypeID, AccountStatusID, Balance, OpeningDate
----------------------------------------------------------------------
✅ addresses       →  1,222 rows ×   4 cols
   Columns: AddressID, Street, City, Country
----------------------------------------------------------------------
✅ branches        →     50 rows ×   3 cols
   Columns: BranchID, BranchName, AddressID
----------------------------------------------------------------------
✅ customer_types  →      3 rows ×   2 cols
   Columns: CustomerTypeID, TypeName
----------------------------------------------------------------------
✅ customers       →  1,11

In [15]:
cleaned_data

{'account_statuses':    AccountStatusID StatusName
 0                1     Active
 1                2   Inactive
 2                3     Closed,
 'account_types':    AccountTypeID  TypeName
 0              1  Checking
 1              2   Savings
 2              3   Payroll
 3              4  Business
 4              5     Youth,
 'accounts':       AccountID  CustomerID  AccountTypeID  AccountStatusID   Balance  \
 0        200094       10123              3                1  48348.54   
 1        201108       10077              3                1  35001.41   
 2        201453       10321              3                2  57081.03   
 3        200581       10871              5                1  63164.33   
 4        200003       10765              1                1  58739.64   
 ...         ...         ...            ...              ...       ...   
 1662     200284       10543              5                1  66775.28   
 1663     200937       10297              5                2  917

In [16]:
print("\n📋 Fixing Transaction Types table...")

# Check if TransactionTypeID is datetime
if 'transaction_types' in cleaned_data:
    df = cleaned_data['transaction_types']
    
    # Check if TransactionTypeID is datetime
    if 'datetime' in str(df['TransactionTypeID'].dtype):
        # Convert back to integer
        # Extract the numeric part from the datetime (it's stored as nanoseconds)
        df['TransactionTypeID'] = df['TransactionTypeID'].dt.nanosecond
        # Or if it's stored differently, you might need:
        # df['TransactionTypeID'] = df['TransactionTypeID'].dt.microsecond // 1000
        
        print(f"   ✅ Converted TransactionTypeID from datetime to integer")
        print(f"   Sample: {df['TransactionTypeID'].head().tolist()}")


📋 Fixing Transaction Types table...
   ✅ Converted TransactionTypeID from datetime to integer
   Sample: [1, 2, 3, 4]


In [17]:
print("\n📋 Fixing Transactions table...")

if 'transactions' in cleaned_data:
    df = cleaned_data['transactions']
    
    # Fix TransactionID (convert from datetime to integer)
    if 'datetime' in str(df['TransactionID'].dtype):
        # Extract the numeric part
        df['TransactionID'] = df['TransactionID'].dt.nanosecond
        print(f"   ✅ Converted TransactionID from datetime to integer")
        print(f"   Sample: {df['TransactionID'].head().tolist()}")
    
    # Fix TransactionTypeID (convert from datetime to integer)
    if 'datetime' in str(df['TransactionTypeID'].dtype):
        df['TransactionTypeID'] = df['TransactionTypeID'].dt.nanosecond
        print(f"   ✅ Converted TransactionTypeID from datetime to integer")
    
    # Handle missing TransactionDate (NaT values)
    missing_dates = df['TransactionDate'].isnull().sum()
    if missing_dates > 0:
        print(f"   ⚠️ Found {missing_dates:,} missing TransactionDate values")
        
        # Option 1: Drop rows with missing TransactionDate
        before = len(df)
        df = df.dropna(subset=['TransactionDate'])
        dropped = before - len(df)
        print(f"   ✅ Dropped {dropped:,} rows with missing TransactionDate")


📋 Fixing Transactions table...
   ✅ Converted TransactionID from datetime to integer
   Sample: [681, 846, 293, 397, 750]
   ✅ Converted TransactionTypeID from datetime to integer
   ⚠️ Found 1,000 missing TransactionDate values
   ✅ Dropped 1,000 rows with missing TransactionDate


In [18]:
# FIX ADDRESSES TABLE

if 'addresses' in cleaned_data:
    df = cleaned_data['addresses']
    
    # Check for missing Country
    missing_country = df['Country'].isnull().sum()
    if missing_country > 0:
        print(f"   ⚠️ Found {missing_country:,} missing Country values")
        
        # Fill with 'Unknown' or 'United States' (since most are US)
        df['Country'].fillna('United States', inplace=True)
        print(f"   ✅ Filled missing Country with 'United States'")

   ⚠️ Found 24 missing Country values
   ✅ Filled missing Country with 'United States'


In [19]:
# VERIFY FIXES
for table_name, df in cleaned_data.items():
    if df is None:
        continue
    
    print(f"\n📋 {table_name.upper()}:")
    
    # Check for datetime columns that shouldn't be datetime
    for col in df.columns:
        if 'datetime' in str(df[col].dtype):
            # Check if column name suggests it should be integer
            if any(keyword in col.lower() for keyword in ['id', 'typeid', 'statusid']):
                print(f"   ⚠️ WARNING: {col} is datetime but likely should be integer")
    
    # Check for remaining missing values
    missing = df.isnull().sum().sum()
    if missing > 0:
        print(f"   ⚠️ Remaining missing values: {missing}")
        # Show which columns have missing values
        missing_cols = df.isnull().sum()
        missing_cols = missing_cols[missing_cols > 0]
        for col, count in missing_cols.items():
            print(f"      - {col}: {count:,}")
    else:
        print(f"   ✅ No missing values")


📋 ACCOUNT_STATUSES:
   ✅ No missing values

📋 ACCOUNT_TYPES:
   ✅ No missing values

📋 ACCOUNTS:
   ⚠️ Remaining missing values: 33
      - OpeningDate: 33

📋 ADDRESSES:
   ⚠️ Remaining missing values: 74
      - Street: 24
      - City: 26
      - Country: 24

📋 BRANCHES:
   ✅ No missing values

📋 CUSTOMER_TYPES:
   ✅ No missing values

📋 CUSTOMERS:
   ⚠️ Remaining missing values: 77
      - FirstName: 22
      - LastName: 23
      - DateOfBirth: 32

📋 LOAN_STATUSES:
   ✅ No missing values

📋 LOANS:
   ⚠️ Remaining missing values: 12
      - StartDate: 6
      - EstimatedEndDate: 6

📋 TRANSACTION_TYPES:
   ✅ No missing values

📋 TRANSACTIONS:
   ⚠️ Remaining missing values: 1000
      - TransactionDate: 1,000


In [ ]:



# FIX ACCOUNTS - Missing OpeningDate


print("\n📋 FIXING ACCOUNTS (OpeningDate missing)")
print("-" * 40)

if 'accounts' in cleaned_data:
    df = cleaned_data['accounts']
    
    missing_count = df['OpeningDate'].isnull().sum()
    if missing_count > 0:
        print(f"   ⚠️ Found {missing_count:,} missing OpeningDate values")
        
        # Option 1: Fill with mode (most common date)
        mode_date = df['OpeningDate'].mode()
        if not mode_date.empty:
            df['OpeningDate'].fillna(mode_date[0], inplace=True)
            print(f"   ✅ Filled with mode: {mode_date[0]}")
        else:
            # Option 2: Fill with a default date (e.g., 2020-01-01)
            default_date = pd.Timestamp('2020-01-01')
            df['OpeningDate'].fillna(default_date, inplace=True)
            print(f"   ✅ Filled with default date: {default_date.date()}")
        
        print(f"   ✅ Remaining missing: {df['OpeningDate'].isnull().sum()}")


# FIX ADDRESSES - Missing Street, City, Country


print("\n📋 FIXING ADDRESSES (Street, City, Country missing)")
print("-" * 40)

if 'addresses' in cleaned_data:
    df = cleaned_data['addresses']
    
    # Fix Street
    missing_street = df['Street'].isnull().sum()
    if missing_street > 0:
        print(f"   ⚠️ Found {missing_street:,} missing Street values")
        mode_street = df['Street'].mode()
        if not mode_street.empty:
            df['Street'].fillna(mode_street[0], inplace=True)
            print(f"   ✅ Filled Street with mode: {mode_street[0]}")
        else:
            df['Street'].fillna('Unknown Street', inplace=True)
            print(f"   ✅ Filled Street with 'Unknown Street'")
    
    # Fix City
    missing_city = df['City'].isnull().sum()
    if missing_city > 0:
        print(f"   ⚠️ Found {missing_city:,} missing City values")
        mode_city = df['City'].mode()
        if not mode_city.empty:
            df['City'].fillna(mode_city[0], inplace=True)
            print(f"   ✅ Filled City with mode: {mode_city[0]}")
        else:
            df['City'].fillna('Unknown City', inplace=True)
            print(f"   ✅ Filled City with 'Unknown City'")
    
    # Fix Country
    missing_country = df['Country'].isnull().sum()
    if missing_country > 0:
        print(f"   ⚠️ Found {missing_country:,} missing Country values")
        # Since most are United States, use that as default
        df['Country'].fillna('United States', inplace=True)
        print(f"   ✅ Filled Country with 'United States'")
    
    print(f"   ✅ Remaining missing: {df.isnull().sum().sum()}")


# FIX CUSTOMERS - Missing FirstName, LastName, DateOfBirth


print("\n📋 FIXING CUSTOMERS (FirstName, LastName, DateOfBirth missing)")
print("-" * 40)

if 'customers' in cleaned_data:
    df = cleaned_data['customers']
    
    # Fix FirstName
    missing_first = df['FirstName'].isnull().sum()
    if missing_first > 0:
        print(f"   ⚠️ Found {missing_first:,} missing FirstName values")
        mode_first = df['FirstName'].mode()
        if not mode_first.empty:
            df['FirstName'].fillna(mode_first[0], inplace=True)
            print(f"   ✅ Filled FirstName with mode: {mode_first[0]}")
        else:
            df['FirstName'].fillna('Unknown', inplace=True)
            print(f"   ✅ Filled FirstName with 'Unknown'")
    
    # Fix LastName
    missing_last = df['LastName'].isnull().sum()
    if missing_last > 0:
        print(f"   ⚠️ Found {missing_last:,} missing LastName values")
        mode_last = df['LastName'].mode()
        if not mode_last.empty:
            df['LastName'].fillna(mode_last[0], inplace=True)
            print(f"   ✅ Filled LastName with mode: {mode_last[0]}")
        else:
            df['LastName'].fillna('Unknown', inplace=True)
            print(f"   ✅ Filled LastName with 'Unknown'")
    
    # Fix DateOfBirth
    missing_dob = df['DateOfBirth'].isnull().sum()
    if missing_dob > 0:
        print(f"   ⚠️ Found {missing_dob:,} missing DateOfBirth values")
        # Fill with median date (middle of date range)
        if not df['DateOfBirth'].empty:
            # Calculate median date
            min_date = df['DateOfBirth'].min()
            max_date = df['DateOfBirth'].max()
            if pd.notna(min_date) and pd.notna(max_date):
                median_date = min_date + (max_date - min_date) / 2
                df['DateOfBirth'].fillna(median_date, inplace=True)
                print(f"   ✅ Filled DateOfBirth with median: {median_date.date()}")
            else:
                default_date = pd.Timestamp('1980-01-01')
                df['DateOfBirth'].fillna(default_date, inplace=True)
                print(f"   ✅ Filled DateOfBirth with default: {default_date.date()}")
        else:
            default_date = pd.Timestamp('1980-01-01')
            df['DateOfBirth'].fillna(default_date, inplace=True)
            print(f"   ✅ Filled DateOfBirth with default: {default_date.date()}")
    
    print(f"   ✅ Remaining missing: {df.isnull().sum().sum()}")


# FIX LOANS - Missing StartDate, EstimatedEndDate


print("\n📋 FIXING LOANS (StartDate, EstimatedEndDate missing)")
print("-" * 40)

if 'loans' in cleaned_data:
    df = cleaned_data['loans']
    
    # Fix StartDate
    missing_start = df['StartDate'].isnull().sum()
    if missing_start > 0:
        print(f"   ⚠️ Found {missing_start:,} missing StartDate values")
        mode_start = df['StartDate'].mode()
        if not mode_start.empty:
            df['StartDate'].fillna(mode_start[0], inplace=True)
            print(f"   ✅ Filled StartDate with mode: {mode_start[0]}")
        else:
            default_date = pd.Timestamp('2021-01-01')
            df['StartDate'].fillna(default_date, inplace=True)
            print(f"   ✅ Filled StartDate with default: {default_date.date()}")
    
    # Fix EstimatedEndDate
    missing_end = df['EstimatedEndDate'].isnull().sum()
    if missing_end > 0:
        print(f"   ⚠️ Found {missing_end:,} missing EstimatedEndDate values")
        mode_end = df['EstimatedEndDate'].mode()
        if not mode_end.empty:
            df['EstimatedEndDate'].fillna(mode_end[0], inplace=True)
            print(f"   ✅ Filled EstimatedEndDate with mode: {mode_end[0]}")
        else:
            # Calculate based on StartDate + 2 years if StartDate exists
            default_date = pd.Timestamp('2024-01-01')
            df['EstimatedEndDate'].fillna(default_date, inplace=True)
            print(f"   ✅ Filled EstimatedEndDate with default: {default_date.date()}")
    
    print(f"   ✅ Remaining missing: {df.isnull().sum().sum()}")


# FIX TRANSACTIONS - Missing TransactionDate


print("\n📋 FIXING TRANSACTIONS (TransactionDate missing)")
print("-" * 40)

if 'transactions' in cleaned_data:
    df = cleaned_data['transactions']
    
    missing_date = df['TransactionDate'].isnull().sum()
    if missing_date > 0:
        print(f"   ⚠️ Found {missing_date:,} missing TransactionDate values")
        
        # Option 1: Drop rows with missing TransactionDate
        before = len(df)
        df = df.dropna(subset=['TransactionDate'])
        dropped = before - len(df)
        print(f"   ✅ Dropped {dropped:,} rows with missing TransactionDate")
        
        # Option 2: Fill with mode (uncomment if you prefer)
        # mode_date = df['TransactionDate'].mode()
        # if not mode_date.empty:
        #     df['TransactionDate'].fillna(mode_date[0], inplace=True)
        #     print(f"   ✅ Filled TransactionDate with mode: {mode_date[0]}")
        
        print(f"   ✅ Remaining missing: {df['TransactionDate'].isnull().sum()}")
        
        # Update the cleaned_data with the fixed dataframe
        cleaned_data['transactions'] = df


# FINAL VERIFICATION


print("\n" + "="*80)
print("✅ FINAL VERIFICATION - ALL TABLES")
print("="*80)

all_clean = True

for table_name, df in cleaned_data.items():
    if df is None:
        continue
    
    missing = df.isnull().sum().sum()
    duplicates = df.duplicated().sum()
    
    # Check for outliers in numeric columns
    numeric_cols = df.select_dtypes(include=[np.number]).columns
    outliers_detected = {}
    
    for col in numeric_cols:
        Q1 = df[col].quantile(0.25)
        Q3 = df[col].quantile(0.75)
        IQR = Q3 - Q1
        if IQR != 0:
            lower = Q1 - 1.5 * IQR
            upper = Q3 + 1.5 * IQR
            outlier_count = len(df[(df[col] < lower) | (df[col] > upper)])
            if outlier_count > 0:
                outliers_detected[col] = outlier_count
    
    issues = []
    
    if missing > 0:
        issues.append(f"⚠️ {missing} missing values")
        all_clean = False
    
    if duplicates > 0:
        issues.append(f"🔄 {duplicates} duplicates")
        all_clean = False
    
    if outliers_detected:
        outlier_str = ", ".join([f"{col}({count})" for col, count in outliers_detected.items()])
        issues.append(f"📊 outliers in: {outlier_str}")
        all_clean = False
    
    status = "✅" if not issues else "⚠️"
    issue_text = f" - {', '.join(issues)}" if issues else " - ✅ ALL CLEAN!"
    
    print(f"{status} {table_name:15s} → {df.shape[0]:>6,} rows × {df.shape[1]:>3} cols{issue_text}")

if all_clean:
    print("\n🎉 ALL TABLES ARE COMPLETELY CLEAN!")
else:
    print("\n⚠️ Some issues remain. Check the details above.")


#  SAVE THE FINALLY CLEANED DATA


print("\n" + "="*80)
print("💾 SAVING FINALLY CLEANED DATA")
print("="*80)

output_path = '../data/processed/'
from pathlib import Path
Path(output_path).mkdir(parents=True, exist_ok=True)

for name, df in cleaned_data.items():
    if df is not None:
        filepath = os.path.join(output_path, f"{name}.csv")
        df.to_csv(filepath, index=False)
        print(f"   ✓ Saved {name} to {filepath}")

print("\n" + "="*80)
print("🎉 ALL DATA CLEANED AND SAVED SUCCESSFULLY!")
print("="*80)


🔧 FIXING REMAINING DATA ISSUES

📋 FIXING ACCOUNTS (OpeningDate missing)
----------------------------------------
   ⚠️ Found 33 missing OpeningDate values
   ✅ Filled with mode: 2018-04-19 00:00:00
   ✅ Remaining missing: 33

📋 FIXING ADDRESSES (Street, City, Country missing)
----------------------------------------
   ⚠️ Found 24 missing Street values
   ✅ Filled Street with mode: Bitting
   ⚠️ Found 26 missing City values
   ✅ Filled City with mode: Country Club Hills
   ⚠️ Found 24 missing Country values
   ✅ Filled Country with 'United States'
   ✅ Remaining missing: 74

📋 FIXING CUSTOMERS (FirstName, LastName, DateOfBirth missing)
----------------------------------------
   ⚠️ Found 22 missing FirstName values
   ✅ Filled FirstName with mode: Van
   ⚠️ Found 23 missing LastName values
   ✅ Filled LastName with mode: Harmon
   ⚠️ Found 32 missing DateOfBirth values
   ✅ Filled DateOfBirth with median: 1993-04-16
   ✅ Remaining missing: 77

📋 FIXING LOANS (StartDate, EstimatedEndDa

In [22]:
"""
🔧 FINAL FIX - PROPERLY HANDLE REMAINING MISSING VALUES
"""

print("\n" + "="*80)
print("🔧 FINAL FIX - PROPERLY HANDLING ALL MISSING VALUES")
print("="*80)

# ============================================================
# 1. FIX ACCOUNTS - Properly fill OpeningDate
# ============================================================

print("\n📋 FIXING ACCOUNTS (OpeningDate missing)")
print("-" * 40)

if 'accounts' in cleaned_data:
    df = cleaned_data['accounts']
    
    # Fill missing OpeningDate with mode
    mode_date = df['OpeningDate'].mode()[0] if not df['OpeningDate'].mode().empty else pd.Timestamp('2020-01-01')
    df.loc[df['OpeningDate'].isnull(), 'OpeningDate'] = mode_date
    
    print(f"   ✅ Filled {df['OpeningDate'].isnull().sum()} missing with mode: {mode_date}")

# ============================================================
# 2. FIX ADDRESSES - Properly fill Street, City, Country
# ============================================================

print("\n📋 FIXING ADDRESSES (Street, City, Country missing)")
print("-" * 40)

if 'addresses' in cleaned_data:
    df = cleaned_data['addresses']
    
    # Fill Street with mode
    mode_street = df['Street'].mode()[0] if not df['Street'].mode().empty else 'Unknown Street'
    df.loc[df['Street'].isnull(), 'Street'] = mode_street
    
    # Fill City with mode
    mode_city = df['City'].mode()[0] if not df['City'].mode().empty else 'Unknown City'
    df.loc[df['City'].isnull(), 'City'] = mode_city
    
    # Fill Country with 'United States'
    df.loc[df['Country'].isnull(), 'Country'] = 'United States'
    
    print(f"   ✅ Filled Street with mode: {mode_street}")
    print(f"   ✅ Filled City with mode: {mode_city}")
    print(f"   ✅ Filled Country with 'United States'")

# ============================================================
# 3. FIX CUSTOMERS - Properly fill FirstName, LastName, DateOfBirth
# ============================================================

print("\n📋 FIXING CUSTOMERS (FirstName, LastName, DateOfBirth missing)")
print("-" * 40)

if 'customers' in cleaned_data:
    df = cleaned_data['customers']
    
    # Fill FirstName with mode
    mode_first = df['FirstName'].mode()[0] if not df['FirstName'].mode().empty else 'Unknown'
    df.loc[df['FirstName'].isnull(), 'FirstName'] = mode_first
    
    # Fill LastName with mode
    mode_last = df['LastName'].mode()[0] if not df['LastName'].mode().empty else 'Unknown'
    df.loc[df['LastName'].isnull(), 'LastName'] = mode_last
    
    # Fill DateOfBirth with median date
    if not df['DateOfBirth'].empty:
        min_date = df['DateOfBirth'].min()
        max_date = df['DateOfBirth'].max()
        if pd.notna(min_date) and pd.notna(max_date):
            median_date = min_date + (max_date - min_date) / 2
            df.loc[df['DateOfBirth'].isnull(), 'DateOfBirth'] = median_date
            print(f"   ✅ Filled DateOfBirth with median: {median_date.date()}")
        else:
            default_date = pd.Timestamp('1980-01-01')
            df.loc[df['DateOfBirth'].isnull(), 'DateOfBirth'] = default_date
            print(f"   ✅ Filled DateOfBirth with default: {default_date.date()}")
    
    print(f"   ✅ Filled FirstName with mode: {mode_first}")
    print(f"   ✅ Filled LastName with mode: {mode_last}")

# ============================================================
# 4. FIX LOANS - Properly fill StartDate, EstimatedEndDate
# ============================================================

print("\n📋 FIXING LOANS (StartDate, EstimatedEndDate missing)")
print("-" * 40)

if 'loans' in cleaned_data:
    df = cleaned_data['loans']
    
    # Fill StartDate with mode
    mode_start = df['StartDate'].mode()[0] if not df['StartDate'].mode().empty else pd.Timestamp('2021-01-01')
    df.loc[df['StartDate'].isnull(), 'StartDate'] = mode_start
    
    # Fill EstimatedEndDate with mode
    mode_end = df['EstimatedEndDate'].mode()[0] if not df['EstimatedEndDate'].mode().empty else pd.Timestamp('2024-01-01')
    df.loc[df['EstimatedEndDate'].isnull(), 'EstimatedEndDate'] = mode_end
    
    print(f"   ✅ Filled StartDate with mode: {mode_start}")
    print(f"   ✅ Filled EstimatedEndDate with mode: {mode_end}")

# ============================================================
# 5. REMOVE DUPLICATES FROM ALL TABLES
# ============================================================

print("\n📋 REMOVING DUPLICATES FROM ALL TABLES")
print("-" * 40)

for table_name, df in cleaned_data.items():
    if df is None:
        continue
    
    dup_count = df.duplicated().sum()
    if dup_count > 0:
        before = len(df)
        df.drop_duplicates(inplace=True)
        removed = before - len(df)
        print(f"   ✅ {table_name}: Removed {removed:,} duplicate rows")
        cleaned_data[table_name] = df
    else:
        print(f"   ✅ {table_name}: No duplicates found")

# ============================================================
# 6. FINAL VERIFICATION
# ============================================================

print("\n" + "="*80)
print("✅ FINAL VERIFICATION - ALL TABLES")
print("="*80)

all_clean = True

for table_name, df in cleaned_data.items():
    if df is None:
        continue
    
    missing = df.isnull().sum().sum()
    duplicates = df.duplicated().sum()
    
    # Check for outliers in numeric columns
    numeric_cols = df.select_dtypes(include=[np.number]).columns
    outliers_detected = {}
    
    for col in numeric_cols:
        Q1 = df[col].quantile(0.25)
        Q3 = df[col].quantile(0.75)
        IQR = Q3 - Q1
        if IQR != 0:
            lower = Q1 - 1.5 * IQR
            upper = Q3 + 1.5 * IQR
            outlier_count = len(df[(df[col] < lower) | (df[col] > upper)])
            if outlier_count > 0:
                outliers_detected[col] = outlier_count
    
    issues = []
    
    if missing > 0:
        issues.append(f"⚠️ {missing} missing values")
        all_clean = False
    
    if duplicates > 0:
        issues.append(f"🔄 {duplicates} duplicates")
        all_clean = False
    
    if outliers_detected:
        outlier_str = ", ".join([f"{col}({count})" for col, count in outliers_detected.items()])
        issues.append(f"📊 outliers in: {outlier_str}")
        all_clean = False
    
    status = "✅" if not issues else "⚠️"
    issue_text = f" - {', '.join(issues)}" if issues else " - ✅ ALL CLEAN!"
    
    print(f"{status} {table_name:15s} → {df.shape[0]:>6,} rows × {df.shape[1]:>3} cols{issue_text}")

if all_clean:
    print("\n🎉 ALL TABLES ARE COMPLETELY CLEAN!")
else:
    print("\n⚠️ Some issues remain. Check the details above.")

# ============================================================
# 7. SAVE THE FINALLY CLEANED DATA
# ============================================================

print("\n" + "="*80)
print("💾 SAVING FINALLY CLEANED DATA")
print("="*80)

output_path = '../data/processed/'
from pathlib import Path
Path(output_path).mkdir(parents=True, exist_ok=True)

for name, df in cleaned_data.items():
    if df is not None:
        filepath = os.path.join(output_path, f"{name}.csv")
        df.to_csv(filepath, index=False)
        print(f"   ✓ Saved {name} to {filepath}")

print("\n" + "="*80)
print("🎉 ALL DATA CLEANED AND SAVED SUCCESSFULLY!")
print("="*80)

# ============================================================
# 8. DISPLAY SAMPLE OF CLEANED DATA
# ============================================================

print("\n🔍 SAMPLE OF CLEANED DATA")
print("="*80)

for name, df in cleaned_data.items():
    if df is not None and not df.empty:
        print(f"\n📋 {name.upper()} (first 3 rows):")
        # Show only first 3 rows and first 5 columns for readability
        display_df = df.iloc[:3, :5]
        print(display_df.to_string())
        print("-" * 50)


🔧 FINAL FIX - PROPERLY HANDLING ALL MISSING VALUES

📋 FIXING ACCOUNTS (OpeningDate missing)
----------------------------------------
   ✅ Filled 0 missing with mode: 2018-04-19 00:00:00

📋 FIXING ADDRESSES (Street, City, Country missing)
----------------------------------------
   ✅ Filled Street with mode: Bitting
   ✅ Filled City with mode: Country Club Hills
   ✅ Filled Country with 'United States'

📋 FIXING CUSTOMERS (FirstName, LastName, DateOfBirth missing)
----------------------------------------
   ✅ Filled DateOfBirth with median: 1993-04-16
   ✅ Filled FirstName with mode: Van
   ✅ Filled LastName with mode: Harmon

📋 FIXING LOANS (StartDate, EstimatedEndDate missing)
----------------------------------------
   ✅ Filled StartDate with mode: 2021-04-05 00:00:00
   ✅ Filled EstimatedEndDate with mode: 2026-03-31 00:00:00

📋 REMOVING DUPLICATES FROM ALL TABLES
----------------------------------------
   ✅ account_statuses: No duplicates found
   ✅ account_types: No duplicates f